# LeetCode 285 — Inorder Successor in BST bằng Python

Link đề: https://leetcode.com/problems/inorder-successor-in-bst/ · Độ khó: **Medium** (Premium) · Chủ đề: Binary Search Tree, Tree, DFS, Inorder

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, x):
#         self.val = x
#         self.left = None
#         self.right = None
class Solution:
    def inorderSuccessor(self, root: 'TreeNode', p: 'TreeNode') -> 'Optional[TreeNode]':
        ...
```

Ghi chú: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (lấy mọi giá trị của cây rồi chọn giá trị nhỏ nhất lớn hơn `p.val`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong một **cây nhị phân tìm kiếm (BST)**, tìm **nút kế nhiệm theo thứ tự giữa (inorder successor)** của nút `p` — nút có **khóa nhỏ nhất lớn hơn `p.val`**; nếu không có thì trả `None`.
2. Nắm **duyệt inorder** của BST cho dãy **tăng dần**, nên successor chính là **phần tử đứng ngay sau `p`** trong dãy đó.
3. Đi từ cách **liệt kê cả dãy inorder** `O(n)` → **inorder lặp dừng sớm** → **tận dụng tính chất BST, đi từ gốc xuống** `O(h)` thời gian và `O(1)` bộ nhớ → **phân tích hai trường hợp** (`p` có con phải / không có) → **đệ quy**.
4. Hiểu vì sao khi **đi sang trái** ta **ghi nhớ nút hiện tại làm ứng viên** successor.
5. Tránh các lỗi kinh điển: trả **giá trị** thay vì **nút**, quên trường hợp **không có con phải** (successor là tổ tiên), dùng sai dấu `<`/`<=`, và nhầm "successor" với "cha của `p`".

## 2. Đề bài & ví dụ

Cho `root` của một cây nhị phân tìm kiếm và một nút `p` thuộc cây, trả về **nút kế nhiệm theo thứ tự giữa** của `p` trong BST. Nếu `p` **không có** successor thì trả `null`.

**Successor** của nút `p` là nút có **khóa nhỏ nhất lớn hơn `p.val`**.

**Ví dụ 1**

- Input: `root = [2,1,3], p = 1`
- Output: `2`
- Giải thích: successor của `1` là `2`. Lưu ý cả `p` và giá trị trả về đều có kiểu `TreeNode`.

**Ví dụ 2**

- Input: `root = [5,3,6,2,4,null,null,1], p = 6`
- Output: `null`
- Giải thích: `6` là khóa lớn nhất nên không có successor.

**Ràng buộc:** số nút trong cây thuộc `[1, 10^4]`; `-10^5 <= Node.val <= 10^5`; mọi nút có giá trị **đôi một khác nhau**.

### Dựng cây và chọn nút `p`

Ta tự dựng `TreeNode`, `build_tree` (danh sách level-order kiểu LeetCode, `None` là nút rỗng) và hàm `find_node(root, val)` để lấy **nút** `p` theo giá trị (vì đề cho `p` là một nút của cây, không phải số).
#### Code demo

In [1]:
from typing import Optional, List
from collections import deque
import random
import time


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: List[Optional[int]]) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def find_node(root, val):
    node = root
    while node and node.val != val:
        node = node.left if val < node.val else node.right
    return node


examples = [
    ([2, 1, 3], 1, 2),
    ([5, 3, 6, 2, 4, None, None, 1], 6, None),
]
for values, pval, expected in examples:
    root = build_tree(values)
    print(f"root={values}, p={pval} (nút có tồn tại: {find_node(root, pval) is not None}) -> kỳ vọng {expected}")

root=[2, 1, 3], p=1 (nút có tồn tại: True) -> kỳ vọng 2
root=[5, 3, 6, 2, 4, None, None, 1], p=6 (nút có tồn tại: True) -> kỳ vọng None


## 3. Kiến thức nền

### 3.1. BST và duyệt inorder

BST: với mọi nút, các giá trị cây con trái `<` nút `<` các giá trị cây con phải. **Duyệt inorder** (trái, gốc, phải) của BST cho dãy **tăng dần**. Vì vậy successor của `p` là phần tử **kế tiếp** `p` trong dãy inorder.
#### Code demo

In [2]:
def inorder_list(root):
    out, stack, node = [], [], root
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        out.append(node.val)
        node = node.right
    return out


root = build_tree([5, 3, 6, 2, 4, None, None, 1])
print(inorder_list(root))          # [1, 2, 3, 4, 5, 6]: tăng dần

[1, 2, 3, 4, 5, 6]


### 3.2. Tìm kiếm trong BST theo tính chất thứ tự

So `x` với `node.val`: nhỏ hơn thì sang **trái**, lớn hơn thì sang **phải**. Mỗi bước loại một nửa (nếu cây cân bằng): `O(h)` với `h` là chiều cao. Đây là cốt lõi của lời giải tối ưu.
#### Code demo

In [3]:
node = root
path = []
target = 4
while node:
    path.append(node.val)
    if target < node.val:
        node = node.left
    elif target > node.val:
        node = node.right
    else:
        break
print("đường đi tìm", target, ":", path)        # [5, 3, 4]

đường đi tìm 4 : [5, 3, 4]


### 3.3. Hai trường hợp của successor

Gọi `p` là nút cần tìm successor.

1. **`p` có con phải:** successor là **nút trái nhất** (nhỏ nhất) trong **cây con phải** của `p`.
2. **`p` không có con phải:** successor là **tổ tiên thấp nhất** mà `p` nằm trong **cây con trái** của nó (tổ tiên đầu tiên mà ta "đi sang trái" khi xuống từ gốc tới `p`). Nếu không có tổ tiên như vậy (p là lớn nhất) thì `None`.
#### Code demo

In [4]:
r = build_tree([5, 3, 6, 2, 4, None, None, 1])
p = find_node(r, 3)                      # có con phải (4): successor = trái nhất của cây con phải = 4
leftmost = p.right
while leftmost.left:
    leftmost = leftmost.left
print("successor(3) =", leftmost.val)

p = find_node(r, 4)                      # không có con phải: successor là tổ tiên mà 4 nằm ở cây con TRÁI của nó
path_up = []
node = r
while node.val != 4:
    path_up.append(node.val)
    node = node.left if 4 < node.val else node.right
print("tổ tiên của 4 (từ gốc):", path_up, "-> tổ tiên cuối cùng mà ta rẽ trái = 5")

successor(3) = 4
tổ tiên của 4 (từ gốc): [5, 3] -> tổ tiên cuối cùng mà ta rẽ trái = 5


### 3.4. Ghi nhớ "ứng viên cuối cùng khi rẽ trái"

Khi xuống từ gốc tìm vị trí của khóa `p.val`: mỗi lần `p.val < node.val` ta rẽ **trái** và `node` là một **ứng viên** successor (nó lớn hơn `p`); mỗi lần ta rẽ **phải** thì `node` nhỏ hơn hoặc bằng `p` nên không phải. Ứng viên **cuối cùng** (thấp nhất, nhỏ nhất) là successor. Cách này **không cần phân tích hai trường hợp** vì khi `p` có con phải, ta đi tiếp xuống bên phải và các ứng viên lớn hơn nằm sâu hơn.
#### Code demo

In [5]:
x = 4
candidate, node = None, r
while node:
    if x < node.val:
        candidate = node
        node = node.left
    else:
        node = node.right
print("successor của", x, "=", candidate.val if candidate else None)    # 5

successor của 4 = 5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Cây `[5,3,6,2,4,null,null,1]`, dãy inorder `1,2,3,4,5,6`.

- Successor của `1` là `2`; của `2` là `3`; của `3` là `4`; của `4` là `5`; của `5` là `6`; của `6` là `None`.
- `3` có con phải `4`: successor là nút trái nhất của cây con phải `{4}` = `4`.
- `4` không có con phải: đi từ gốc `5 → 3 → 4`; lần rẽ **trái** duy nhất là ở `5` (vì `4 < 5`) ⇒ successor `5`.
- `6`: từ gốc `5 → 6`, chỉ rẽ phải ⇒ không có ứng viên ⇒ `None`.

**Cách nghĩ:** successor = "nút nhỏ nhất vẫn lớn hơn `p`"; trong BST ta tìm được nó bằng một lần đi từ gốc xuống, ghi lại nút cuối cùng mà ta phải rẽ trái.

## 5. Cách 1 — Brute Force / baseline: liệt kê cả dãy inorder rồi chọn phần tử kế tiếp

**Ý tưởng.** Duyệt inorder lấy danh sách **nút**; tìm vị trí của `p`; trả nút ở vị trí kế tiếp (hoặc `None`).
#### Code demo

In [6]:
def successor_inorder_list(root, p):
    nodes, stack, node = [], [], root
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        nodes.append(node)
        node = node.right
    for i, n in enumerate(nodes):
        if n is p:
            return nodes[i + 1] if i + 1 < len(nodes) else None
    return None


for values, pval, expected in examples:
    root = build_tree(values)
    res = successor_inorder_list(root, find_node(root, pval))
    print(values, pval, "->", res.val if res else None, "(kỳ vọng", expected, ")")

[2, 1, 3] 1 -> 2 (kỳ vọng 2 )
[5, 3, 6, 2, 4, None, None, 1] 6 -> None (kỳ vọng None )


**Độ phức tạp (Cách 1).** Gọi `n` là số nút: **Time `O(n)`**, **Space `O(n)`** (danh sách các nút và stack). Không tận dụng BST để loại bớt phần cây.

**Nút thắt.** Ta duyệt cả cây dù chỉ cần một nút. Tính chất thứ tự BST cho phép **đi thẳng** một đường từ gốc.

## 6. Ý tưởng thuật toán / bước đột phá

**Đi từ gốc xuống, ghi nhớ ứng viên:** với `succ = None`, lặp `while root`: nếu `p.val < root.val` thì `root` có thể là successor ⇒ `succ = root` và đi **trái** (để tìm ứng viên nhỏ hơn mà vẫn `> p.val`); ngược lại (`p.val >= root.val`) đi **phải**. Kết thúc, `succ` là đáp án.

Mỗi bước đi xuống một tầng ⇒ **`O(h)`** thời gian, **`O(1)`** bộ nhớ. Điểm mấu chốt: tại mỗi bước, ta loại hẳn một cây con không thể chứa đáp án tốt hơn.

## 7. Các cách tiếp theo

### Cách 2 — Inorder lặp, dừng ngay sau khi gặp `p`

**Giải thích.** Duyệt inorder bằng stack; dùng cờ `found`: khi đã gặp `p`, nút inorder kế tiếp là đáp án và dừng. Vẫn `O(h + k)` với `k` là số nút trước `p` trong inorder.
#### Code demo

In [7]:
def successor_inorder_early_stop(root, p):
    stack, node = [], root
    found = False
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        if found:
            return node                         # nút inorder ngay sau p
        if node is p:
            found = True
        node = node.right
    return None


for values, pval, expected in examples:
    root = build_tree(values)
    res = successor_inorder_early_stop(root, find_node(root, pval))
    print(values, pval, "->", res.val if res else None, "(kỳ vọng", expected, ")")

[2, 1, 3] 1 -> 2 (kỳ vọng 2 )
[5, 3, 6, 2, 4, None, None, 1] 6 -> None (kỳ vọng None )


**Độ phức tạp Cách 2:** **Time `O(n)`** worst case (nếu `p` ở cuối), **Space `O(h)`** cho stack.

### Cách 3 — Đi từ gốc xuống và nhớ ứng viên (cách nộp chuẩn)

**Giải thích.** Như mục 6.
#### Code demo

In [8]:
def successor_bst_walk(root, p):
    succ = None
    node = root
    while node:
        if p.val < node.val:
            succ = node                         # node lớn hơn p: ứng viên; thử tìm ứng viên nhỏ hơn ở bên trái
            node = node.left
        else:
            node = node.right                   # node <= p: successor phải nằm ở bên phải
    return succ


for values, pval, expected in examples:
    root = build_tree(values)
    res = successor_bst_walk(root, find_node(root, pval))
    print(values, pval, "->", res.val if res else None, "(kỳ vọng", expected, ")")

[2, 1, 3] 1 -> 2 (kỳ vọng 2 )
[5, 3, 6, 2, 4, None, None, 1] 6 -> None (kỳ vọng None )


In [9]:
# Code nộp LeetCode cho Cách 3
class Solution:
    def inorderSuccessor(self, root: 'TreeNode', p: 'TreeNode') -> 'Optional[TreeNode]':
        ans = None
        while root:
            if root.val > p.val:
                ans = root
                root = root.left
            else:
                root = root.right
        return ans


r0 = build_tree([2, 1, 3])
print(Solution().inorderSuccessor(r0, find_node(r0, 1)).val)    # 2

2


**Độ phức tạp Cách 3:** **Time `O(h)`** (`h = O(log n)` cây cân bằng, `O(n)` cây lệch), **Space `O(1)`**.

### Cách 4 — Phân tích hai trường hợp (con phải / tổ tiên)

**Giải thích.** Nếu `p.right` tồn tại: trả nút **trái nhất** của `p.right`. Ngược lại: đi từ gốc xuống tìm `p`; mỗi lần rẽ **trái** ghi lại ứng viên; trả ứng viên cuối cùng.
#### Code demo

In [10]:
def successor_two_cases(root, p):
    if p.right:
        node = p.right
        while node.left:
            node = node.left
        return node
    ancestor = None
    node = root
    while node is not p:
        if p.val < node.val:
            ancestor = node                      # tổ tiên mà p nằm ở cây con trái của nó
            node = node.left
        else:
            node = node.right
    return ancestor


for values, pval, expected in examples:
    root = build_tree(values)
    res = successor_two_cases(root, find_node(root, pval))
    print(values, pval, "->", res.val if res else None, "(kỳ vọng", expected, ")")

[2, 1, 3] 1 -> 2 (kỳ vọng 2 )
[5, 3, 6, 2, 4, None, None, 1] 6 -> None (kỳ vọng None )


**Độ phức tạp Cách 4:** **Time `O(h)`**, **Space `O(1)`**. Nêu rõ cấu trúc của đáp án, nhưng dài hơn Cách 3.

### Cách 5 — Đệ quy

**Giải thích.** `succ(root, p)`: nếu `root` rỗng trả `None`; nếu `root.val <= p.val` thì đáp án nằm ở **cây con phải**; ngược lại thử cây con **trái**: nếu cây con trái có ứng viên thì dùng, không thì chính `root`.
#### Code demo

In [11]:
def successor_recursive(root, p):
    if root is None:
        return None
    if root.val <= p.val:
        return successor_recursive(root.right, p)
    left = successor_recursive(root.left, p)
    return left if left else root


for values, pval, expected in examples:
    root = build_tree(values)
    res = successor_recursive(root, find_node(root, pval))
    print(values, pval, "->", res.val if res else None, "(kỳ vọng", expected, ")")

[2, 1, 3] 1 -> 2 (kỳ vọng 2 )
[5, 3, 6, 2, 4, None, None, 1] 6 -> None (kỳ vọng None )


**Độ phức tạp Cách 5:** **Time `O(h)`**, **Space `O(h)`** (ngăn xếp đệ quy; với cây lệch `n = 10^4` có thể chạm giới hạn đệ quy mặc định).

> Cách 3 là bản nộp ngắn gọn và tối ưu; Cách 4 giúp hiểu cấu trúc; Cách 1 dùng làm baseline.

## 8. TRACE từng bước

Ta trace Cách 3 trên cây `[5,3,6,2,4,null,null,1]` cho `p = 4`, `p = 3` và `p = 6`.
#### Code demo

In [12]:
def trace_walk(root, pval):
    p = find_node(root, pval)
    print(f"tìm successor của p={pval}")
    succ, node = None, root
    while node:
        if p.val < node.val:
            print(f"  node={node.val}: {pval} < {node.val} -> ứng viên = {node.val}, rẽ TRÁI")
            succ = node
            node = node.left
        else:
            print(f"  node={node.val}: {pval} >= {node.val} -> không phải ứng viên, rẽ PHẢI")
            node = node.right
    print("  kết quả:", succ.val if succ else None)
    return succ


tree = build_tree([5, 3, 6, 2, 4, None, None, 1])
for pv in (4, 3, 6):
    trace_walk(tree, pv)
    print()

tìm successor của p=4
  node=5: 4 < 5 -> ứng viên = 5, rẽ TRÁI
  node=3: 4 >= 3 -> không phải ứng viên, rẽ PHẢI
  node=4: 4 >= 4 -> không phải ứng viên, rẽ PHẢI
  kết quả: 5

tìm successor của p=3
  node=5: 3 < 5 -> ứng viên = 5, rẽ TRÁI
  node=3: 3 >= 3 -> không phải ứng viên, rẽ PHẢI
  node=4: 3 < 4 -> ứng viên = 4, rẽ TRÁI
  kết quả: 4

tìm successor của p=6
  node=5: 6 >= 5 -> không phải ứng viên, rẽ PHẢI
  node=6: 6 >= 6 -> không phải ứng viên, rẽ PHẢI
  kết quả: None



Bảng tóm tắt (`p = 3`):

| `node` | so sánh | ứng viên sau bước | di chuyển |
|---|---|---|---|
| `5` | `3 < 5` | `5` | trái |
| `3` | `3 >= 3` | `5` | phải |
| `4` | `3 < 4` | `4` | trái |
| `None` | dừng | `4` | — |

Ứng viên cuối cùng `4` là successor của `3` (đúng với "trái nhất của cây con phải").

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3. Đúng `class Solution`, đúng `inorderSuccessor(self, root, p)`, trả về **nút** (hoặc `None`).

In [13]:
class Solution:
    def inorderSuccessor(self, root: 'TreeNode', p: 'TreeNode') -> 'Optional[TreeNode]':
        ans = None
        while root:
            if root.val > p.val:          # root lớn hơn p: có thể là successor, thử ứng viên nhỏ hơn ở bên trái
                ans = root
                root = root.left
            else:                         # root <= p: successor chỉ có thể ở bên phải
                root = root.right
        return ans


sol = Solution()
for values, pval, expected in examples:
    root = build_tree(values)
    res = sol.inorderSuccessor(root, find_node(root, pval))
    assert (res.val if res else None) == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Invariant:** tại mỗi lúc, `ans` là nút nhỏ nhất đã duyệt qua mà `> p.val`, và successor thật nằm trong `{ans}` ∪ (cây con của `root` hiện tại).
- **Rẽ trái khi `root.val > p.val`:** `root` là ứng viên hợp lệ; mọi nút lớn hơn `root` trong cây con phải của `root` đều **lớn hơn `root`** nên không tốt hơn; successor thật (nhỏ nhất `> p.val`) nằm trong `{root}` ∪ cây con trái của `root`.
- **Rẽ phải khi `root.val <= p.val`:** `root` và cả cây con trái của nó đều `<= p.val` (BST), nên không thể là successor; chỉ cây con phải còn khả năng.
- **Kết thúc:** `root` thành `None` sau tối đa `h` bước; `ans` là nút nhỏ nhất `> p.val` hoặc `None` nếu không có.
- **Khớp hai trường hợp:** nếu `p` có con phải thì ta đi qua `p` rồi rẽ phải, các ứng viên lớn hơn `p` nằm trong cây con phải; ứng viên cuối cùng là trái nhất. Nếu `p` không có con phải, ứng viên cuối cùng là tổ tiên cuối cùng ta rẽ trái.
- **`<=` (không `<`):** khi `root.val == p.val` (đang đứng tại chính `p`) phải đi **phải**, vì chính `p` không phải successor của nó.

## 11. Phân tích Time / Space Complexity

Gọi `n` số nút, `h` chiều cao.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê inorder | `O(n)` | `O(n)` |
| 2. Inorder dừng sớm | `O(n)` worst | `O(h)` |
| 3. Đi từ gốc, nhớ ứng viên | `O(h)` | `O(1)` |
| 4. Hai trường hợp | `O(h)` | `O(1)` |
| 5. Đệ quy | `O(h)` | `O(h)` |

`h = O(log n)` với cây cân bằng, `O(n)` với cây lệch (`n = 10^4`). Với `n = 10^4`, mọi cách đều nhanh; điểm khác là **bậc**: `O(h)` có thể chỉ cỡ `14` bước thay vì `10^4`.

## 12. Bộ test

`oracle` là brute force độc lập: lấy **mọi giá trị** của cây bằng duyệt stack tự quản, rồi chọn **giá trị nhỏ nhất lớn hơn `p.val`** (hoặc `None`), so sánh **giá trị** (và danh tính nút) với kết quả mọi cách trên **mọi nút `p`** của cây ngẫu nhiên.
#### Code demo

In [14]:
def all_values(root):
    out, stack = [], ([root] if root else [])
    while stack:
        node = stack.pop()
        out.append(node.val)
        for c in (node.left, node.right):
            if c:
                stack.append(c)
    return out


def all_nodes(root):
    out, stack = [], ([root] if root else [])
    while stack:
        node = stack.pop()
        out.append(node)
        for c in (node.left, node.right):
            if c:
                stack.append(c)
    return out


def oracle(root, p):
    bigger = [v for v in all_values(root) if v > p.val]
    return min(bigger) if bigger else None


def bst_from_values(values):
    root = None
    for x in values:
        node = TreeNode(x)
        if root is None:
            root = node
            continue
        cur = root
        while True:
            if x < cur.val:
                if cur.left is None:
                    cur.left = node
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = node
                    break
                cur = cur.right
    return root


impls = {
    "inorder_list": successor_inorder_list,
    "early_stop": successor_inorder_early_stop,
    "bst_walk": successor_bst_walk,
    "two_cases": successor_two_cases,
    "recursive": successor_recursive,
    "submit": sol.inorderSuccessor,
}

for values, pval, expected in examples:
    root = build_tree(values)
    p = find_node(root, pval)
    assert oracle(root, p) == expected
    for key, fn in impls.items():
        res = fn(root, p)
        assert (res.val if res else None) == expected, (key, values, pval)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [15]:
edge_cases = [
    ([1], 1, None, "một nút"),
    ([2, 1], 1, 2, "con trái: successor là cha"),
    ([2, 1], 2, None, "gốc là lớn nhất"),
    ([1, None, 2], 1, 2, "gốc có con phải"),
    ([1, None, 2], 2, None, "lá phải cuối cùng"),
    ([2, 1, 3], 2, 3, "gốc có con phải: trái nhất của cây con phải"),
    ([5, 3, 6, 2, 4, None, None, 1], 4, 5, "không con phải: tổ tiên"),
    ([5, 3, 6, 2, 4, None, None, 1], 1, 2, "lá trái nhất"),
    ([5, 3, 6, 2, 4, None, None, 1], 5, 6, "gốc, con phải là lá"),
    ([20, 10, 30, 5, 15, 25, 35, None, None, 12, 18], 15, 18, "con phải có con trái: trái nhất của cây con phải"),
    ([20, 10, 30, 5, 15, 25, 35, None, None, 12, 18], 18, 20, "không con phải, tổ tiên cách xa"),
    ([-100000, None, 100000], -100000, 100000, "giá trị biên của đề"),
]
for values, pval, expected, label in edge_cases:
    root = build_tree(values)
    p = find_node(root, pval)
    assert oracle(root, p) == expected, (label, oracle(root, p), expected)
    for key, fn in impls.items():
        res = fn(root, p)
        assert (res.val if res else None) == expected, (key, label, res.val if res else None, expected)
    print("PASS", label)

PASS một nút
PASS con trái: successor là cha
PASS gốc là lớn nhất
PASS gốc có con phải
PASS lá phải cuối cùng
PASS gốc có con phải: trái nhất của cây con phải
PASS không con phải: tổ tiên
PASS lá trái nhất
PASS gốc, con phải là lá
PASS con phải có con trái: trái nhất của cây con phải
PASS không con phải, tổ tiên cách xa
PASS giá trị biên của đề


### 12.2. Tính chất và phản ví dụ

- Successor của `p` luôn có giá trị `> p.val` và là **nhỏ nhất** trong các giá trị như vậy; nút lớn nhất không có successor.
- Chuỗi successor liên tiếp từ nút **nhỏ nhất** đi qua **mọi** nút theo thứ tự tăng dần (khớp dãy inorder).
- **Phản ví dụ** cho lỗi "successor là cha": với `p` là con phải của `3` trong cây `[5,3,6,2,4]` thì successor của `4` là `5` chứ không phải cha `3`.
#### Code demo

In [16]:
rng = random.Random(285)
for _ in range(300):
    n = rng.randint(1, 40)
    values = rng.sample(range(-100, 100), n)
    root = bst_from_values(values)
    nodes = sorted(all_nodes(root), key=lambda x: x.val)
    # chuỗi successor từ nút nhỏ nhất đi qua mọi nút theo thứ tự tăng dần
    cur, walked = nodes[0], []
    while cur:
        walked.append(cur.val)
        cur = sol.inorderSuccessor(root, cur)
    assert walked == [x.val for x in nodes]
print("PASS tính chất: chuỗi successor khớp dãy inorder (300 cây ngẫu nhiên)")


def sai_cha(root, p):
    parent, node = None, root
    while node is not p:
        parent = node
        node = node.left if p.val < node.val else node.right
    return parent                                    # SAI: successor không phải luôn là cha


t = build_tree([5, 3, 6, 2, 4])
p4 = find_node(t, 4)
print("phản ví dụ: successor của 4 đúng", sol.inorderSuccessor(t, p4).val, "| trả cha:", sai_cha(t, p4).val)
assert sai_cha(t, p4).val != sol.inorderSuccessor(t, p4).val

PASS tính chất: chuỗi successor khớp dãy inorder (300 cây ngẫu nhiên)
phản ví dụ: successor của 4 đúng 5 | trả cha: 3


### 12.3. Đối chiếu ngẫu nhiên (có seed) và cây lệch sâu
#### Code demo

In [17]:
rng = random.Random(4285)
checked = 0
for trial in range(400):
    n = rng.randint(1, 60)
    values = rng.sample(range(-1000, 1000), n)
    if trial % 4 == 0:
        values = sorted(values)                        # cây lệch phải
    elif trial % 4 == 1:
        values = sorted(values, reverse=True)           # cây lệch trái
    root = bst_from_values(values)
    for p in all_nodes(root):
        truth = oracle(root, p)
        for key, fn in impls.items():
            res = fn(root, p)
            assert (res.val if res else None) == truth, (key, values, p.val)
        checked += 1
print(f"PASS {checked} cặp (cây, nút p) ngẫu nhiên gồm cả cây lệch: mọi cách khớp oracle")

# Cây rất sâu (n = 3000 lệch phải), chỉ dùng các cách không đệ quy
chain = bst_from_values(list(range(3000)))
p = find_node(chain, 1500)
t0 = time.perf_counter()
assert successor_bst_walk(chain, p).val == 1501 and sol.inorderSuccessor(chain, p).val == 1501
print(f"cây lệch 3000 nút: Cách 3 tốn {(time.perf_counter() - t0) * 1000:.2f} ms")

PASS 12275 cặp (cây, nút p) ngẫu nhiên gồm cả cây lệch: mọi cách khớp oracle
cây lệch 3000 nút: Cách 3 tốn 0.16 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trả về giá trị thay vì nút
#### Code sai

In [18]:
def sai_tra_gia_tri(root, p):
    ans = None
    while root:
        if root.val > p.val:
            ans = root.val                              # SAI: đề yêu cầu trả NÚT (TreeNode)
            root = root.left
        else:
            root = root.right
    return ans


t = build_tree([2, 1, 3])
print("đúng:", type(sol.inorderSuccessor(t, find_node(t, 1))).__name__, "| sai:", type(sai_tra_gia_tri(t, find_node(t, 1))).__name__)

đúng: TreeNode | sai: int


**Vì sao sai:** kết quả phải là **nút** (có `.val`, `.left`, `.right`), không phải số. Trên LeetCode kiểu trả về sai bị từ chối hoặc không in đúng.

### Lỗi 2: Chỉ xét con phải, quên trường hợp không có con phải
#### Code sai

In [19]:
def sai_chi_con_phai(root, p):
    if p.right:
        node = p.right
        while node.left:
            node = node.left
        return node
    return None                                         # SAI: quên tổ tiên


t = build_tree([5, 3, 6, 2, 4, None, None, 1])
for pv in (3, 4):
    res = sai_chi_con_phai(t, find_node(t, pv))
    print(f"p={pv}: đúng", sol.inorderSuccessor(t, find_node(t, pv)).val, "| sai:", res.val if res else None)

p=3: đúng 4 | sai: 4
p=4: đúng 5 | sai: None


**Vì sao sai:** khi `p` không có con phải, successor là **tổ tiên** (`4 → 5`). Cách sửa: đi từ gốc ghi nhớ ứng viên (Cách 3) hoặc dùng con trỏ cha.

### Lỗi 3: Dùng `>=` hoặc `<` sai chiều (đứng tại chính `p`)
#### Code sai

In [20]:
def sai_dau_bang(root, p):
    ans = None
    while root:
        if root.val >= p.val:                           # SAI: khi root là chính p, p bị coi là successor của nó
            ans = root
            root = root.left
        else:
            root = root.right
    return ans


t = build_tree([2, 1, 3])
print("p=1 đúng:", sol.inorderSuccessor(t, find_node(t, 1)).val, "| sai:", sai_dau_bang(t, find_node(t, 1)).val)

p=1 đúng: 2 | sai: 1


**Vì sao sai:** successor phải **lớn hơn hẳn** `p.val`; dùng `>=` làm chính `p` thành ứng viên. Quy tắc: `root.val > p.val`.

### Lỗi 4: Sang trái khi nhỏ hơn mà không ghi nhớ ứng viên (đi như tìm kiếm thường)
#### Code sai

In [21]:
def sai_khong_nho_ung_vien(root, p):
    node = root
    while node and node is not p:
        node = node.left if p.val < node.val else node.right
    # sau khi tìm thấy p, chỉ nhìn con phải; mất thông tin tổ tiên đã đi qua
    if node is None or node.right is None:
        return None
    node = node.right
    while node.left:
        node = node.left
    return node


t = build_tree([5, 3, 6, 2, 4])
p4 = find_node(t, 4)
print("p=4 đúng:", sol.inorderSuccessor(t, p4).val, "| sai:", sai_khong_nho_ung_vien(t, p4))

p=4 đúng: 5 | sai: None


**Vì sao sai:** thông tin "tổ tiên cuối cùng ta rẽ trái" chỉ có khi đi từ gốc xuống; vứt nó đi thì không biết successor khi `p` không có con phải.

### Lỗi 5: Quên `None` (cây/ nút rỗng)
Nếu `p` là nút lớn nhất, kết quả phải là `None`; code truy cập `.val` của kết quả mà không kiểm tra sẽ gây `AttributeError`. Hãy luôn kiểm tra `if res is not None`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**successor / predecessor trong BST**" → **đi từ gốc xuống** và ghi nhớ ứng viên: successor khi rẽ **trái**, predecessor khi rẽ **phải**.
- Tìm **nhỏ nhất lớn hơn** / **lớn nhất nhỏ hơn** một khóa (ceiling / floor) trong BST: cùng khung này.
- Có **con trỏ cha**: successor = trái nhất của cây con phải; nếu không có con phải thì **đi lên** cho tới khi ta là con **trái** của cha.
- Dùng tính chất BST để `O(h)` thay vì duyệt `O(n)`.
- So sánh **nghiêm ngặt** `root.val > p.val`.
- Luôn thử: lá phải cuối cùng (`None`), gốc có con phải, lá trái nhất, cây lệch.

## 15. Biến thể của bài

1. **Inorder Successor in BST II** (LeetCode 510): có con trỏ `parent`, không cho gốc.
2. **Predecessor**: nút lớn nhất nhỏ hơn `p`.
3. **Ceiling / Floor** của một khóa `x` (không nhất thiết có trong cây).
4. **Closest BST Value** (270), **Kth Smallest in a BST** (230).
5. **Insert / Delete in BST** (701, 450): xóa nút hai con dùng successor thay thế.
#### Code demo

In [22]:
def predecessor(root, p):
    ans = None
    while root:
        if root.val < p.val:
            ans = root
            root = root.right
        else:
            root = root.left
    return ans


def ceiling(root, x):
    """Nút nhỏ nhất có val >= x."""
    ans = None
    while root:
        if root.val >= x:
            ans = root
            root = root.left
        else:
            root = root.right
    return ans


t = build_tree([5, 3, 6, 2, 4, None, None, 1])
print(predecessor(t, find_node(t, 4)).val, predecessor(t, find_node(t, 1)), ceiling(t, 4).val, ceiling(t, 7))   # 3 None 4 None

3 None 4 None


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê inorder | duyệt cả cây rồi lấy phần tử kế | `O(n)` | `O(n)` | baseline, oracle |
| 2. Inorder dừng sớm | dừng ngay sau `p` | `O(n)` worst | `O(h)` | cây bất kỳ (không là BST) cũng dùng được |
| 3. Từ gốc, nhớ ứng viên | rẽ trái ghi ứng viên | `O(h)` | `O(1)` | **cách nộp chuẩn** |
| 4. Hai trường hợp | con phải / tổ tiên | `O(h)` | `O(1)` | giải thích cấu trúc |
| 5. Đệ quy | `root.val <= p.val` sang phải, ngược lại ưu tiên trái | `O(h)` | `O(h)` | viết ngắn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `inorder_list(build_tree([4, 2, 6, 1, 3, 5, 7]))`; (b) successor của `3` trong cây đó; (c) successor của `7`; (d) successor của `4`.

**Bài 2 (tính bằng tay).** Với cây `[20, 10, 30, 5, 15, 25, 35, None, None, 12, 18]`, tìm successor của `10`, `18`, `35` và giải thích mỗi trường hợp thuộc dạng nào.

**Bài 3 (điền TODO).** Hoàn thiện `successor_todo(root, p)` theo Cách 3.
#### Code demo

In [23]:
def successor_todo(root, p):
    ans = None
    # TODO 1: while root
    # TODO 2: nếu root.val > p.val: ans = root; root = root.left
    # TODO 3: ngược lại root = root.right
    # TODO 4: trả ans
    pass


# Tự làm xong rồi mới mở test:
# for values, pval, expected in examples:
#     root = build_tree(values); res = successor_todo(root, find_node(root, pval))
#     assert (res.val if res else None) == expected

**Bài 4 (tự viết lại).** Viết `kth_successor(root, p, k)` trả về nút successor thứ `k` (lặp lại successor `k` lần); kiểm tra với `sorted` các giá trị.

**Bài 5 (biến thể / mini challenge).** Viết `successor_with_parent(p)` cho cây **có con trỏ cha** (`node.parent`): nếu `p.right` thì trái nhất của nó; ngược lại đi lên cho tới khi nút hiện tại là con **trái** của cha, trả cha (hoặc `None`). Tự gắn `parent` rồi đối chiếu với `oracle`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: inorder của BST tăng dần. Bài 2: `10` có con phải; `18` là lá phải của `15`; `35` lớn nhất. Bài 5: đi lên cho tới khi là con trái.
- **Mức 2 (vừa):** Bài 1: `[1,2,3,4,5,6,7]`, `4`, `None`, `5`. Bài 2: `10 → 12` (trái nhất của cây con phải `{15,12,18}`); `18 → 20` (đi lên: `18` là con phải của `15`, `15` là con phải của `10`, `10` là con trái của `20`); `35 → None`. Bài 3: khung Cách 3.
- **Mức 3 (sâu):** Bài 4: lặp `for _ in range(k): p = sol.inorderSuccessor(root, p); nếu None thì dừng`. Bài 5: `while p.parent and p.parent.right is p: p = p.parent; return p.parent`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
t17 = build_tree([4, 2, 6, 1, 3, 5, 7])
assert inorder_list(t17) == [1, 2, 3, 4, 5, 6, 7]
assert sol.inorderSuccessor(t17, find_node(t17, 3)).val == 4
assert sol.inorderSuccessor(t17, find_node(t17, 7)) is None
assert sol.inorderSuccessor(t17, find_node(t17, 4)).val == 5

# Bài 2
t20 = build_tree([20, 10, 30, 5, 15, 25, 35, None, None, 12, 18])
assert sol.inorderSuccessor(t20, find_node(t20, 10)).val == 12
assert sol.inorderSuccessor(t20, find_node(t20, 18)).val == 20
assert sol.inorderSuccessor(t20, find_node(t20, 35)) is None


# Bài 3
def successor_todo_ans(root, p):
    ans = None
    while root:
        if root.val > p.val:
            ans = root
            root = root.left
        else:
            root = root.right
    return ans


for values, pval, expected in examples:
    root = build_tree(values)
    res = successor_todo_ans(root, find_node(root, pval))
    assert (res.val if res else None) == expected


# Bài 4
def kth_successor(root, p, k):
    for _ in range(k):
        p = sol.inorderSuccessor(root, p)
        if p is None:
            return None
    return p


rng = random.Random(16)
for _ in range(100):
    values = rng.sample(range(-50, 50), rng.randint(1, 20))
    root = bst_from_values(values)
    ordered = sorted(values)
    start = rng.randrange(len(ordered))
    k = rng.randint(1, 5)
    res = kth_successor(root, find_node(root, ordered[start]), k)
    assert (res.val if res else None) == (ordered[start + k] if start + k < len(ordered) else None)


# Bài 5
def attach_parents(root, parent=None):
    stack = [(root, None)]
    while stack:
        node, par = stack.pop()
        if node:
            node.parent = par
            stack.append((node.left, node))
            stack.append((node.right, node))


def successor_with_parent(p):
    if p.right:
        p = p.right
        while p.left:
            p = p.left
        return p
    while p.parent and p.parent.right is p:
        p = p.parent
    return p.parent


for _ in range(100):
    values = rng.sample(range(-50, 50), rng.randint(1, 25))
    root = bst_from_values(values)
    attach_parents(root)
    for p in all_nodes(root):
        res = successor_with_parent(p)
        assert (res.val if res else None) == oracle(root, p)
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đi từ gốc xuống, ghi nhớ ứng viên)

```python
class Solution:
    def inorderSuccessor(self, root: 'TreeNode', p: 'TreeNode') -> 'Optional[TreeNode]':
        ans = None
        while root:
            if root.val > p.val:
                ans = root
                root = root.left
            else:
                root = root.right
        return ans
```

**Checklist**

- [ ] Trả về **nút** (`TreeNode`) hoặc `None`, không phải số?
- [ ] `root.val > p.val` (nghiêm ngặt): rẽ **trái** và ghi `ans = root`; ngược lại rẽ **phải**?
- [ ] Xử lý đủ hai dạng: `p` có con phải / không có (tổ tiên)?
- [ ] Nút lớn nhất trả `None`?
- [ ] Đã thử cây một nút, cây lệch, lá phải cuối cùng, con phải có con trái?
- [ ] Đã đối chiếu với brute force "giá trị nhỏ nhất lớn hơn `p.val`" trên mọi nút của cây ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def inorderSuccessor(self, root, p)`?